# 08 — Offline Testing (zero API keys)

## Objectives

- Run agents **without any network or API key** using `TestModel` and `Agent('test')`
- Script exact model behavior with `FunctionModel`
- Assert on what actually happened with `capture_run_messages`
- Swap a "real" agent's model per-test with `agent.override`
- Prove nothing leaks to the network with the `ALLOW_MODEL_REQUESTS` kill switch

> This notebook runs **keyless** — it is exactly what the repo's CI runs on every push.

In [ ]:
# ── SETUP — run this first in every notebook ─────────────────────────────────
import os
import sys

import logfire
from dotenv import find_dotenv, load_dotenv

# 1) Environment: walk up from notebooks/ to the repo root and load .env
load_dotenv(find_dotenv())

# 2) Observability: no Logfire account yet? This still works — data only ships
#    when a token exists (`uv run logfire auth`, or a LOGFIRE_TOKEN in .env).
logfire.configure(send_to_logfire="if-token-present")
logfire.instrument_pydantic_ai()  # v2: instrumentation is explicit — Agent(instrument=True) is gone

MODEL = os.getenv("MODEL", "openai:gpt-6.1-sol")

print("kernel :", sys.executable)  # should end in learningdantic/.venv/bin/python
print("MODEL  :", MODEL)

## Why

Real models are slow, flaky, and cost money — a terrible substrate for tests. Pydantic AI
ships its own test doubles, so your agent's *plumbing* (tools, validators, history, wiring)
is verified deterministically, for free, and your live-model runs stay for learning and demos.
Docs: [pydantic.dev/docs/ai/guides/testing](https://pydantic.dev/docs/ai/guides/testing/)

## `Agent('test')` and `TestModel` — the keyless fake

`Agent('test')` is a built-in shortcut for `Agent(TestModel())`. By default `TestModel`
calls **every** registered tool, then returns a JSON record of the calls — perfect for
checking that tools are wired up, with zero network.

In [ ]:
from pydantic_ai import Agent, RunContext
from pydantic_ai.models.test import TestModel

calc_agent = Agent("test", instructions="You do arithmetic exclusively with your tools.")


@calc_agent.tool_plain
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b


@calc_agent.tool
async def multiply(ctx: RunContext[None], a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b


r1 = await calc_agent.run("What is 6 times 7?")
print("1. default calls ALL tools:", repr(r1.output))

r2 = await calc_agent.run("What is 6 times 7?", model=TestModel(call_tools=["multiply"]))
print("2. call_tools=[...] narrows:", repr(r2.output))

> ✅ **Checkpoint** — outputs like `'{"add":0,"mul":0}'` and `'{"multiply":0}'`: TestModel
> exercised your tools with placeholder arguments (0s) and recorded the calls. It proves
> *wiring*, not intelligence — for behavior, you script it:

## `FunctionModel` — a scripted model

Your function receives the messages and returns whatever `ModelResponse` you want —
a tool call, or text. You control the entire "model" side of the conversation:

In [ ]:
from pydantic_ai.messages import ModelResponse, TextPart, ToolCallPart
from pydantic_ai.models.function import FunctionModel


def scripted_model(messages, info):
    """First request: call add_numbers(2, 3). After the tool result returns: answer with text."""
    tool_ran = any(
        type(part).__name__ == "ToolReturnPart" for message in messages for part in message.parts
    )
    if tool_ran:
        return ModelResponse(parts=[TextPart(content="The sum is 5.")])
    return ModelResponse(parts=[ToolCallPart(tool_name="add_numbers", args={"a": 2, "b": 3})])


scripted_agent = Agent(FunctionModel(scripted_model), instructions="arith")


@scripted_agent.tool_plain
def add_numbers(a: int, b: int) -> int:
    """Add two integers."""
    return a + b


r = await scripted_agent.run("what is 2+3?")
print("3. scripted flow output:", repr(r.output))

## Assert on the real exchange with `capture_run_messages`

The whole point: your tests can assert on *exactly what the agent did* — which tools,
in which order, with what results:

In [ ]:
from pydantic_ai import capture_run_messages
from pydantic_ai.messages import ToolReturnPart

with capture_run_messages() as messages:
    r = await scripted_agent.run("what is 2+3?")

kinds = [type(part).__name__ for message in messages for part in message.parts]
print("4. captured flow:", kinds)

assert kinds == ["UserPromptPart", "ToolCallPart", "ToolReturnPart", "TextPart"], kinds
tool_returns = [p for m in messages for p in m.parts if isinstance(p, ToolReturnPart)]
assert tool_returns[0].content == 5, tool_returns
print("   agent asked for add, ran it (result 5), then answered — exactly as scripted")

> ✅ **Checkpoint** — the assertion passed: prompt → tool call → tool return → text.
> You just tested an agent's full loop with no model, no key, no network, in milliseconds.

## `override` + the kill switch — how real test suites stay offline

Your production agents carry real model strings, but a keyless `Agent('openai:...')`
won't even construct — construction checks for the API key. So for a fully keyless demo
we build a "real" agent with an explicit provider and a **dummy key**; the mechanism is
identical to production code:

- `agent.override(model=TestModel())` — swap the model for the duration of a block
- `override_allow_model_requests(False)` — a context manager that makes any *real* model
  request raise immediately, so a test suite can prove it never touches the network
  (test doubles like `TestModel`/`FunctionModel` are exempt).

In [ ]:
from pydantic_ai import models
from pydantic_ai.models.openai import OpenAIResponsesModel
from pydantic_ai.providers.openai import OpenAIProvider

live_agent = Agent(
    OpenAIResponsesModel("gpt-6.1-sol", provider=OpenAIProvider(api_key="dummy-not-real")),
    instructions="never actually reached",
)

with live_agent.override(model=TestModel(call_tools=[])):
    r = await live_agent.run("anything")  # served by the override, not the network
print("5. override kept the run offline:", repr(r.output))

with models.override_allow_model_requests(False):
    try:
        await live_agent.run("this would be a real, billable request")
    except RuntimeError as e:
        print("6. blocked as designed:", e)
    else:
        msg = "a real request went through — the kill switch failed!"
        raise AssertionError(msg)

> ✅ **Checkpoint** — line 5 ran via the override; line 6 raised `RuntimeError` **before**
> any network traffic. Wrap your whole test suite in that context manager and unbillable
> becomes *provably* unbillable.

## Pinning exact outputs

`TestModel(custom_output_text=...)` makes the fake return exactly that string — handy for
snapshot-style tests where you want a known, stable answer:

In [ ]:
pinned_agent = Agent(TestModel(custom_output_text="EXACT EXPECTED SENTENCE"), instructions="x")

r = await pinned_agent.run("say the thing")
assert r.output == "EXACT EXPECTED SENTENCE"
print("7. pinned output matched exactly:", repr(r.output))

## How CI uses this

Open [`.github/workflows/ci.yml`](../.github/workflows/ci.yml) — on every push, GitHub Actions
runs `pytest` on **this notebook** and **00**, with **zero secrets**. If you can clone the repo,
you can verify it works; if CI is green, the plumbing is sound. (Running `uv run pytest`
locally executes *all* notebooks, using your keys.)

## 🏋️ Exercise — the offline proof

Using `scripted_agent`'s pattern, prove all three of these **without any API key**:

1. A **spy counter** on `add` shows the tool really executed (increment inside the function).
2. With `capture_run_messages`, the `ToolCallPart` always arrives *before* the matching
   `ToolReturnPart`.
3. The whole run still works inside `override_allow_model_requests(False)` — test doubles
   are exempt from the kill switch.

Write them as plain `assert` statements that fail loudly if the agent misbehaves.

Your take: [`../solutions/08_offline_assertions.ipynb`](../solutions/08_offline_assertions.ipynb).

## What's next

That's Phase 1 — you can build, observe, and test agents. The roadmap for Phase 2:
toolsets & MCP servers (09), multi-agent orchestration (10), evals (11), serving with
FastAPI (12), SQLModel-backed agents (13), and the capstone support assistant (14).